# StayOps — Chunk Creation & Vector Indexing

1. Load `final_data.xlsx` using `os` paths
2. Rebuild `summary` column in code (skip empty values, strip HTML tags)
3. Split text into section-tagged chunks
4. Embed with **SentenceTransformers** (`BAAI/bge-small-en-v1.5`) and persist in ChromaDB as `collection_{property_id}`

In [4]:
#from __future__ import annotations

import os
import sys
import pandas as pd
from dotenv import load_dotenv
from rag.chunking import chunks_from_summary, build_summary_column, resolve_property_id
from rag.retriever import chroma_client, collection_name, get_or_create_property_collection

load_dotenv(override=True)


FINAL_DATA_PATH = os.getenv("FINAL_DATA_PATH").strip()
CHROMA_PATH = os.getenv("CHROMA_PATH").strip()
EMBED_MODEL = os.getenv("EMBEDDING_MODEL").strip()

print("Excel:", FINAL_DATA_PATH)
print("Chroma:", CHROMA_PATH)
print("Embedding Model:", EMBED_MODEL)

Excel: data/final_data.xlsx
Chroma: chroma_db/UNITS_INFO_CHUNCK
Embedding Model: BAAI/bge-small-en-v1.5


In [5]:
if not os.path.exists(FINAL_DATA_PATH):
    raise FileNotFoundError(f"Missing data file at: {FINAL_DATA_PATH}")

df = pd.read_excel(FINAL_DATA_PATH)

df.head(2)

,property_address1,property_address2,property_building,property_id,property_notes_access,property_notes_general,property_notes_guest_access,property_photos_url,property_state,property_status,...,listing_room_ids,listing_room_numbers,listing_room_bed_ids,listing_room_bed_types,listing_room_bed_quantities,custom_field_values,import_time,date_of_first_scrape,date_of_last_update,date_of_last_scrape
0,NaN,NaN,NaN,910,NaN,NaN,NaN,NaN,NaN,NaN,...,66c30cd399e922187e4f7065--x--66c30cd399e922187...,1--x--2--x--3--x--4--x--5--x--6--x--7--x--8--x--9,66d0af51c611340019a4aacd--x--66d0af51c61134001...,QUEEN_BED--x--QUEEN_BED--x--QUEEN_BED--x--QUEE...,1--x--1--x--1--x--1--x--2--x--5,Â \nÂ Thank you so much for booking! Your door...,NaN,NaT,NaN,NaT
1,NaN,NaN,NaN,925,NaN,NaN,NaN,NaN,NaN,NaN,...,66c30cd099e922187e4f6fa8--x--66c30cd099e922187...,NaN,6700ea406bf9e50010f23892--x--6700ea406bf9e5001...,SINGLE_BED--x--QUEEN_BED--x--SINGLE_BED--x--KI...,1--x--1--x--2--x--1,"Arrival and Parking: \n- Park in the driveway,...",2024-08-16T14:57:41.519Z,2024-12-25 18:50:00,NaN,2024-12-25 18:50:00


## Build Summary Column in Code

Creating sumary column by combining rest of column and its value

In [8]:
df = build_summary_column(df)

print(df["summary"].iloc[0][:700])

property_id: 910; listing_id: 66bf68fa7011f00014648c16; saas_auto_renew: 1.0; cleaning_fee_id: 66bf68fa7011f00014648c17; cleaning_fee_value_type: FIXED; cleaning_fee_formula: 470.0; cleaning_fee_multiplier: PER_STAY; channel_commission_use_account_settings: 1.0; channel_commission_id: 67334bb16d8308d9819747e0; channel_commission_created_at: 2024-11-12T12:36:01.011Z; channel_commission_updated_at: 2024-12-11T07:01:40.069Z; picture_caption: Aerial; picture_thumbnail: https://track-pm.s3.amazonaws.com/gw/image/822ff2f9-e043-42bf-8366-39076c6e06c9; minimum_nights: 1.0; maximum_nights: 365.0; monthly_price_factor: 1.0; weekly_price_factor: 1.0; base_price: 1.0; weekend_base_price: 469.0; currency


In [9]:
sample_id = resolve_property_id(df.iloc[0])
sample_chunks = chunks_from_summary(str(df.iloc[0]["summary"]), sample_id)
print(f"property_id={sample_id} → Generated {len(sample_chunks)} chunks")
for i, chunk in enumerate(sample_chunks[:3]):
    print(f"\n[{i}] Section '{chunk['section']}': {chunk['text'][:200]}...")

property_id=910 → Generated 20 chunks

[0] Section 'profile': Property profile for ID 910. property_id: 910; listing_id: 66bf68fa7011f00014648c16; cleaning_fee_id: 66bf68fa7011f00014648c17; channel_commission_id: 67334bb16d8308d9819747e0; auto_payments_time_rela...

[1] Section 'identity': property_id: 910; listing_id: 66bf68fa7011f00014648c16; cleaning_fee_id: 66bf68fa7011f00014648c17; channel_commission_id: 67334bb16d8308d9819747e0; auto_payments_time_relation_names: AT--x--BEFORE; re...

[2] Section 'other': saas_auto_renew: 1.0; picture_caption: Aerial; picture_thumbnail: https://track-pm.s3.amazonaws.com/gw/image/822ff2f9-e043-42bf-8366-39076c6e06c9; confirmed_during_stay_delay_minutes: 45.0; unconfirme...


## Store Chunks in ChromaDB with SentenceTransformers Embeddings

By using `BAAI/bge-small-en-v1.5` model ,chunk embeddings are getting stored in vector database 

In [ ]:
os.makedirs(CHROMA_PATH, exist_ok=True)
client = chroma_client()

stored_properties = 0
total_chunks = 0

for row_idx, row in df.iterrows():
    try:
        property_id = resolve_property_id(row)
    except ValueError:
        continue
        
    chunks = chunks_from_summary(str(row["summary"]), property_id)
    if not chunks:
        continue
        
    name = collection_name(property_id)

        
    collection = get_or_create_property_collection(property_id)
    collection.add(
        documents=[c["text"] for c in chunks],
        metadatas=[{"section": c["section"], "property_id": c["property_id"]} for c in chunks],
        ids=[f"{property_id}_{i}" for i in range(len(chunks))],
    )
    stored_properties += 1
    total_chunks += len(chunks)
    print(f"Indexed property '{property_id}' (row {row_idx}) ({len(chunks)} chunks)")

print(f"\nDone: Indexed {stored_properties} properties with {total_chunks} total chunks into {CHROMA_PATH}")

Indexed property '910' (row 0) (20 chunks)
Indexed property '925' (row 1) (14 chunks)
Indexed property '930' (row 2) (13 chunks)
Indexed property '953' (row 3) (13 chunks)
Indexed property '1095' (row 4) (13 chunks)
Indexed property '1182' (row 5) (13 chunks)
Indexed property '1188' (row 6) (12 chunks)
Indexed property '1193' (row 7) (9 chunks)
Indexed property '1201' (row 8) (14 chunks)
Indexed property '1204' (row 9) (12 chunks)
Indexed property '453841' (row 10) (5 chunks)
Indexed property '453845' (row 11) (5 chunks)
Indexed property '453886' (row 12) (5 chunks)
Indexed property '453929' (row 13) (5 chunks)
Indexed property '453966' (row 14) (5 chunks)
Indexed property '453990' (row 15) (5 chunks)
Indexed property '453997' (row 16) (5 chunks)
Indexed property '454000' (row 17) (5 chunks)
Indexed property '454007' (row 18) (5 chunks)
Indexed property '454066' (row 19) (5 chunks)

Done: Indexed 20 properties with 183 total chunks into chroma_db/UNITS_INFO_CHUNCK


In [12]:
from rag.embeddings import query_embedding_function

test_property = sample_id
query = "where can guests park and what is the wifi info?"
collection = get_or_create_property_collection(test_property)

q_emb = query_embedding_function()([query])
results = collection.query(query_embeddings=q_emb, n_results=3)

print("Test Query:", query)
for doc, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    print(f"\nDistance: {dist:.3f} | Section: {meta.get('section')}")
    print(doc)

Test Query: where can guests park and what is the wifi info?

Distance: 0.331 | Section: other
there is evidence found of an animal presence during your stay or post check-out, you will be charged additional cleaning fees. - No Smoking Allowed Anywhere Inside or Outside. If evidence is found, you will be charged additional fees. - There are paddleboards, kayaks, lily pad, paddles, and life vests available for guest use. Please put back all equipment back in the same place that you found them after use. -If bringing a boat or jet skis, there are several options for tying off your boat and/or jet skis. Boat and jet skis at property are owners and not for guest use. Boats can be launched at Kingsland Community Park located at 155 Lions Park Road, Kingsland, TX 78639. There is a $10 launch fee - there is no overnight trailer parking at boat launch. Guests may park boat trailers at property. Â Check-Out: - Please leave the house as you found it - nice and tidy. Please return anything used t